# 1. Clone the repo

In [ ]:
!git clone https://github.com/Asaf21S/flow-matching-as-a-layer.git
%cd flow-matching-as-a-layer
!git checkout asaf/stage3_continuation2

In [ ]:
%cd /content/flow-matching-as-a-layer
!git pull --ff-only

# 2. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

# 3. Session paths and imports

In [ ]:
import os
import sys
import logging
import warnings
from pathlib import Path

# Set to True to keep features and results on Drive and reuse them across sessions.
USE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/fmlayer")

REPO_ROOT = Path("/content/flow-matching-as-a-layer")
DATA_ROOT = Path("/content/data")   # always session-local: too big to sync

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    FEATURE_ROOT = DRIVE_ROOT / "features"
    RESULTS_ROOT = DRIVE_ROOT / "results"
else:
    FEATURE_ROOT = Path("/content/features")
    RESULTS_ROOT = Path("/content/results")

for path in (DATA_ROOT, FEATURE_ROOT, RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

os.environ["FMLAYER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["FMLAYER_FEATURE_ROOT"] = str(FEATURE_ROOT)
os.environ["FMLAYER_RESULTS_ROOT"] = str(RESULTS_ROOT)
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.ERROR)

import torch
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("features ->", FEATURE_ROOT)
print("results  ->", RESULTS_ROOT)

Check what is already cached
Stage 3 reads the Stage 1 feature caches and reuses the Stage 1 probes. If the features are already on Drive you can skip cell 4 entirely — no dataset download is needed to train the flow.

In [ ]:
for path in sorted(FEATURE_ROOT.rglob("*.np*")):
    print(f"{path.relative_to(FEATURE_ROOT)}  {path.stat().st_size / 1e6:.1f} MB")

print("\nresults:")
for path in sorted(RESULTS_ROOT.glob("*")):
    count = len(list(path.glob("*"))) if path.is_dir() else ""
    print(f"  {path.name:<22} {count}")

# 4. Datasets, features, subsets

Only needed when the Drive cache is empty. DTD is ~625 MB, FGVC-Aircraft ~2.75 GB.

In [ ]:
from src.fmlayer.data.prepare import prepare_datasets
from src.fmlayer.features.extract import extract_all
from src.fmlayer.data.fewshot import build_all_subsets

prepare_datasets()
extract_all()
build_all_subsets()
print("Datasets, feature caches and subset indices ready.")

# 5. Component self-checks

In [ ]:
from src.fmlayer.train.checks import run_all_checks

run_all_checks()

# 6. The Stage 3 experiments

## 6.0 One-time cache note

In [ ]:
for name in ("curves_fm_stage3", "models_stage3"):
    for path in (RESULTS_ROOT / name).glob("stage3_rolled_ce_T12*"):
        path.unlink()
print("rolled_ce checkpoints cleared; they retrain with the full curve logging")

## 6.1 Main comparison — K = 10, T = 12, seeds 0/1/2

In [ ]:
from src.fmlayer.train.train_fm import run_stage3_main

main_results = run_stage3_main(k=10, seeds=(0, 1, 2), max_epochs=500, verbose=True)

4 configs x 2 cells x 3 seeds = 24 runs. Finished runs load from Drive, so the cell is safe to interrupt and re-run.

## 6.2 Secondary comparison — K = full

In [ ]:
full_results = run_stage3_main(k="full", seeds=(0, 1, 2), max_epochs=500, verbose=True)

## 6.3 Strategy 1: regularising the rolled-out objective
The brief suggests "penalising the displacement between z and z_hat, or the magnitude of the predicted velocities". Both penalties are divided by the mean squared feature norm, so the weights mean the same thing on ResNet-18 and on DINOv2.

In [ ]:
from src.fmlayer.train.train_fm import rolled_regularization_configs, run_stage3_main

regularised = run_stage3_main(
    k=10, seeds=(0,), configs=rolled_regularization_configs(), max_epochs=500, verbose=True
)

## 6.4 Strategy 2: the four knobs the brief asks about
Feature-space step size (target_lr), number of target-improvement steps (target_steps), whether the target update is constrained (target_normalize, which turns the step into a fixed fraction of the feature norm) and how often the targets are recomputed (target_refresh, in epochs; 1 recomputes every batch). The first entry is the main-comparison run itself, so the sweep contains its own reference for free.

In [ ]:
from src.fmlayer.train.train_fm import guided_ablation_configs, run_stage3_main

guided_sweep = run_stage3_main(
    k=10, seeds=(0,), configs=guided_ablation_configs(), max_epochs=500, verbose=True
)

### After a session restart: load, do not re-run

In [ ]:
from src.fmlayer.train.train_fm import MAIN_CELLS, load_stage3_results, main_configs

main_results = load_stage3_results(
    cells=MAIN_CELLS, k_values=(10,), seeds=(0, 1, 2),
    configs=main_configs(), step_counts=(12,),
)

If it reports zero runs, FMLAYER_RESULTS_ROOT is not pointing at the Drive folder the grid was written to — re-run cell 3, or check:

In [ ]:
from src.fmlayer.train.train_fm import stage3_cache_summary

stage3_cache_summary()

# 7. Results, tables and figures

## 7.0 Drop the stale rows from runs.csv
Three method names were written by earlier versions of the tag scheme and describe runs that now live under different names. Left in place, the aggregation counts them as extra configurations.

In [ ]:
import pandas as pd
from src.fmlayer.utils.results import runs_csv_path

STALE = (
    "stage3_standard_guided",
    "stage3_standard_guided_n015",
    "stage3_standard_guided_n15_s1lr1",
)

path = runs_csv_path(RESULTS_ROOT)
rows = pd.read_csv(path)
kept = rows[~rows["method"].isin(STALE)]
kept.to_csv(path, index=False)
print(f"dropped {len(rows) - len(kept)} stale row(s), {len(kept)} left")
kept["method"].value_counts()

## 7.1 The headline table
main_comparison_table puts the frozen probe and every Stage 3 method side by side, with accuracy averaged over seeds and a paired delta: each run is scored against the probe of its own seed, so the spread of the delta is the right error bar. markdown_main_table prints it ready to paste into docs/stage3.md.



In [ ]:
from src.fmlayer.stage3_report import main_comparison_table, markdown_main_table, save_main_table

main_table = main_comparison_table(main_results, k=10, steps=12)
print(markdown_main_table(main_table))
save_main_table(main_table, RESULTS_ROOT)
main_table

Same for the K=full table:

In [ ]:
full_table = main_comparison_table(full_results, k="full", steps=12)
print(markdown_main_table(full_table))

And for the two ablations, whose configuration names are not in the main label map, so pass your own labels:

In [ ]:
from src.fmlayer.train.train_fm import guided_ablation_configs, rolled_regularization_configs

labels = {config.name: config.name for config in guided_ablation_configs()}
print(markdown_main_table(main_comparison_table(guided_sweep, k=10, steps=12, labels=labels)))

labels = {config.name: config.name for config in rolled_regularization_configs()}
print(markdown_main_table(main_comparison_table(regularised, k=10, steps=12, labels=labels)))

## 7.2 Diagnostics — did the flow do anything at all?
An accuracy number cannot tell "the flow collapsed to the identity" apart from "the flow moved points and it hurt". This measures the relative displacement and counts label flips in both directions. Paste this table into the write-up: it is the evidence behind the conclusion.

In [ ]:
from src.fmlayer.train.diagnostics import diagnose_all, print_diagnostics

diagnostics = diagnose_all(main_results, feature_root=FEATURE_ROOT)
print_diagnostics(diagnostics)

Read it like this:

* move near 0 with flip% near 0 -> the flow has collapsed to the identity, and its small negative delta is just approximation error.
* broken much larger than fixed -> the flow is moving points, and the moves are net-harmful.
* fixed larger than broken -> genuinely worth pursuing on the full grid.

## 7.3 The three required figures
One call writes, per cell:

training behaviour — stage3_curves_<dataset>_<encoder>.png: training loss, validation accuracy and validation cross-entropy, one line per method. The two methods minimise different quantities, so only the two right-hand panels compare them fairly.
learned dynamics — viz_<config>_<...>.png per method: training and validation curves, the vector field at t = 0, and the trajectories at T = 12 only.
feature space — flow_comparison_<dataset>_<encoder>_T12.png: the original test features next to the transported features of each method, on one jointly fitted PCA, with the same examples and the same class colours in every panel.

In [ ]:
from src.fmlayer.stage3_report import make_stage3_main_figures

figures = make_stage3_main_figures(
    main_results, k=10, seed=0, steps=12,
    feature_root=FEATURE_ROOT, figures_root=RESULTS_ROOT / "figures",
    show=True, save=True,
)

## 7.4 Export everything the write-up cites

In [ ]:
import shutil
from src.fmlayer.stage3_report import save_stage3_table, stage3_table

save_stage3_table(stage3_table(main_results), RESULTS_ROOT)

REPO_RESULTS = REPO_ROOT / "results"
(REPO_RESULTS / "figures").mkdir(parents=True, exist_ok=True)

for name in ("runs.csv", "stage3_main.csv", "stage3_table.csv"):
    source = RESULTS_ROOT / name
    if source.is_file():
        shutil.copy2(source, REPO_RESULTS / name)

for png in (RESULTS_ROOT / "figures").glob("*.png"):
    shutil.copy2(png, REPO_RESULTS / "figures" / png.name)

print("copied into the repo; commit results/ so the numbers are reproducible")